In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import logging
from Scripts.logger import setup_logger

In [18]:
trane_url = 'https://www.tranetechnologies.com/en/index/news.html'
## works w/ base fetch

##trane_news_url = 'https://investors.tranetechnologies.com/news-and-events/news-releases/default.aspx'
## doesn't work w/ base fetch

trane_article_url = 'https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2025/Trane-Technologies-to-Acquire-Stellar-Energy-Digital-Business/default.aspx'
## works w/ base fetch


### original base fetch method

In [ ]:
def trane_fetch_method(url, logger):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        )
    }
    try:
        response = requests.get(url, headers=headers, verify=False, timeout=30)
        html_doc = response.text
        soup = BeautifulSoup(html_doc, 'html.parser')
        return soup
    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching URL {url}: {e}")
        return None

### Extract htmls

In [22]:
def trane_url_extraction(temp_soup):
    # Find all <a> tags with class newspromo__link
    links = temp_soup.find_all("a", class_="newspromo__link")

    # Extract href attributes
    hrefs = [a['href'] for a in links if 'href' in a.attrs]

    # Keep only links that start with https:
    https_links = [link for link in hrefs if link.startswith("https:")]
    return https_links

### Trane article scraping method

In [ ]:
def trane_scrape_article(url,logger):
    
    soup = trane_fetch_method(url,logger)

    ## extract title
    title=soup.find('h3').get_text().strip()


    ## extract summary / newslinetext 
    full_text = " ".join(p.get_text().strip() for p in soup.find_all('p'))
    result_text = full_text.split("About Trane Technologies")[0].strip()
    result_text = result_text.split("About Trane")[0].strip()

    ## extract date
    dateline = None

    # 1️⃣ Try main module date
    try:
        date_text = soup.find("span", class_="module_date-text").get_text(strip=True)
        dateline = datetime.strptime(date_text, '%B %d, %Y')
    except Exception:
        pass

    # 2️⃣ Fallback to <span class="value">
    if dateline is None:
        spans = soup.find_all("span", class_="value")
        for span in spans:
            text = span.get_text(strip=True)
            # Try multiple formats
            for fmt in ("%b %d, %Y %I:%M %p", "%b %d, %Y", "%B %d, %Y %I:%M %p", "%B %d, %Y"):
                try:
                    text_clean = text.replace(" ET", "")
                    dateline = datetime.strptime(text_clean, fmt)
                    break
                except ValueError:
                    continue
            if dateline is not None:
                break

    # 3️⃣ Default to now if still None
    if dateline is None:
        dateline = datetime.now()

    # Ensure standard format: YYYY-MM-DD HH:MM:SS
    dateline = dateline.replace(microsecond=0)

    split_text = result_text.split('.')
    summary = '.'.join(split_text[:3]) + '.'

    article = {
        'title': title,
        'summary': summary,
        'dateline': dateline, 
        'newslinetext': result_text,
        'attachmenturl': url,
        'source': 'Trane Technologies'
    }
    return article 

# Implementation

In [ ]:
# ## scrape homepage url for article links
# temp_soup = trane_fetch_method(trane_url)

# # # Find all <a> tags with class newspromo__link
# # links = temp_soup.find_all("a", class_="newspromo__link")

# # # Extract href attributes
# # hrefs = [a['href'] for a in links if 'href' in a.attrs]

# # # Keep only links that start with https:
# # https_links = [link for link in hrefs if link.startswith("https:")]


# https_links = trane_url_extraction(temp_soup)

# ## iterate through article links and scrape each articles data

# article_data = pd.DataFrame(columns=['title','summary','dateline','newslinetext','url', 'source'])

# for link in https_links:
#     article_data = pd.concat([article_data, trane_scrape_article(link)], ignore_index=True)

c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2026/Trane-Technologies-to-Present-at-the-Barclays-Industrial-Select-Conference/default.aspx


c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'investors.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2026/Trane-Technologies-to-Present-at-the-Citi-Global-Industrial-Tech-and-Mobility-Conference/default.aspx


C:\Users\203156\AppData\Local\Temp\ipykernel_33372\1249053776.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
C:\Users\203156\AppData\Local\Temp\ipykernel_33372\227074174.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  article_data = pd.concat([article_data, trane_scrape_article(link)], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: Insec

...

C:\Users\203156\AppData\Local\Temp\ipykernel_33372\1249053776.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.trane.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2026/Trane-Technologies-Reports-Strong-Fourth-Quarter-and-Full-Year-2025-Results-Robust-Bookings-and-Backlog-Provide-Strong-Visibility-Entering-2026/default.aspx


C:\Users\203156\AppData\Local\Temp\ipykernel_33372\1249053776.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'investors.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200


C:\Users\203156\AppData\Local\Temp\ipykernel_33372\1249053776.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)


In [ ]:
# ## write to excel
# article_data.to_excel('Data/trane_news.xlsx', index=False)